In [1]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.add(["JuMP", "HiGHS", "Plots"])

  Activating project at `c:\Users\asus\OneDrive - Singapore University of Technology and Design\Documents\GitHub\optimization`
   Resolving package versions...
     Project No packages added to or removed from `C:\Users\asus\OneDrive - Singapore University of Technology and Design\Documents\GitHub\optimization\Project.toml`
    Manifest No packages added to or removed from `C:\Users\asus\OneDrive - Singapore University of Technology and Design\Documents\GitHub\optimization\Manifest.toml`


In [2]:
Pkg.instantiate()

In [3]:
using Random, Printf, Statistics
Random.seed!(2024)

TaskLocalRNG()

In [20]:
#~345 total berths across all terminals. 
#     ~130 are "Large Berths" (LOA ≥ 200m or Depth > 12m)
#     ~215 are "Small Berths" (LOA < 200m)
#   - In dynamic port sch# ================================================================
# REAL-WORLD MARITIME DATA — Port of Singapore
# All values calibrated using MPA Wharves & Berths data + Oceans-X
#
# CONTEXT:
#   - Traffic Demand is taken from Oceans-X (Past 24 hr duration)
#   - "Parking Lots" (Berths): Based on the provided document, there are
#     eduling, since most berths are occupied by 
#     long-stay vessels, we model the *Available Unreserved Lots* or 
#     *Berthing Processing Capacity* per 30-min slot.
#   - Constraints: Large ships CANNOT use small berths. Small ships CAN 
#     use large berths. 
#   - Weather & Tide: "Low Tide" restricts deep-draft large ships.
#     "Squalls / High Wind > 20kts" halts berthing operations (as 
#     noted in the PCS Jurong Island guidelines).
#   - Priority: Large ships (VLCCs, Large Containers) have massive 
#     charter/operating costs, heavily prioritizing them over small ships.
# ================================================================

T = 48    # 48 × 30-min slots covering full 24-hour day (00:00–23:30)
Q = 20    # 20 weather/tide scenarios 
K = 7     # 7 Vessel Classes

println("="^72)
println("PORT OF SINGAPORE (MPA) — MARITIME TRAFFIC & BERTHING SCHEDULING")
println("="^72)

# ----------------------------------------------------------------
# 1. TRAFFIC DEMAND DATA (From Oceans-X)
# ----------------------------------------------------------------
period_label =[@sprintf("%02d:%02d", (i-1)÷2, ((i-1)%2)*30) for i in 1:T]

class_names =["Bulk Carrier", "Tanker", "Container", "Cargo", "Tug/Supply", "Passenger", "Other"]
#1.Bulk Carrier (40,000 GT): Represents a Panamax or Kamsarmax bulk carrier (~75,000 to 82,000 Deadweight Tonnage [DWT])
#2.Tanker (80,000 GT): Represents a Suezmax crude/product tanker (~150,000 DWT).
#3.Container (100,000 GT): Represents a Neo-Panamax container ship (approx. 9,000 to 10,000 TEU). Note: 100k GT is the statistical median for mainline East-West trade routes.
#4.Cargo (20,000 GT): Represents a Handysize general cargo or breakbulk vessel.
#5.Tug/Supply (2,000 GT): Represents an Offshore Support Vessel (OSV) or a large Anchor Handling Tug (AHTS).
#6.Passenger (120,000 GT): Represents a Large Cruise Ship (e.g. Royal Caribbean)
#7.Other (10,000 GT): Miscellaneous vessels like cable layers, research vessels, or large dredgers.

# Boolean mapping: Which classes require a "Large Berth" (LOA ≥ 200m / Deep Draft)
is_large_vessel =[true, true, true, false, false, true, false]

N_Total =[
  # 00:00 00:30 01:00 01:30 02:00 02:30 03:00 03:30
       1,    1,    1,    0,    1,    2,    0,    0,
  # 04:00 04:30 05:00 05:30 06:00 06:30 07:00 07:30
       2,    0,    1,    1,    2,    2,    2,    4,
  # 08:00 08:30 09:00 09:30 10:00 10:30 11:00 11:30
       4,    2,    1,    6,    7,    3,    0,    2,
  # 12:00 12:30 13:00 13:30 14:00 14:30 15:00 15:30
       2,    3,    0,    3,    3,    1,    4,    1,
  # 16:00 16:30 17:00 17:30 18:00 18:30 19:00 19:30
       1,    1,    0,    5,    4,    2,    3,    2,
  # 20:00 20:30 21:00 21:30 22:00 22:30 23:00 23:30
       3,    0,    2,    1,    2,    3,    1,    2
]
# Original Oceans-X Data Arrays
N_Vessels = [  [0,1,0,0,0,2,0,0, 0,0,0,0,0,1,0,1, 1,1,0,2,0,2,0,1, 2,1,0,3,1,1,3,0, 0,1,0,3,1,1,1,2, 1,0,2,0,0,1,1,0], # 1. Bulk
               [1,0,0,0,1,0,0,0, 0,0,0,0,1,1,1,0, 1,0,1,1,2,1,0,0, 0,1,0,0,2,0,0,0, 1,0,0,0,0,1,0,0, 1,0,0,1,1,1,0,1], # 2. Tanker
               [0,0,0,0,0,0,0,0, 0,0,1,1,0,0,0,1, 0,0,0,1,2,0,0,1, 0,0,0,0,0,0,0,0, 0,0,0,1,1,0,1,0, 0,0,0,0,0,1,0,0], # 3. Container
               [0,0,1,0,0,0,0,0, 1,0,0,0,0,0,1,0, 1,0,0,1,0,0,0,0, 0,0,0,0,0,0,0,1, 0,0,0,0,0,0,0,0, 1,0,0,0,0,0,0,0], # 4. Cargo
               [0,0,0,0,0,0,0,0, 1,0,0,0,1,0,0,2, 0,0,0,1,1,0,0,0, 0,0,0,0,0,0,1,0, 0,0,0,1,1,0,1,0, 0,0,0,0,0,0,0,0], # 5. Tug/Supply
               [0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 0,1,0,0,1,0,0,0, 0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0], # 6. Passenger
               [0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0, 1,0,0,0,1,0,0,0, 0,1,0,0,0,0,0,0, 0,0,0,0,1,0,0,0, 0,0,0,0,1,0,0,1]  # 7. Other
]


# Aggregate into Large and Small categories based on typical LOA/Draft
N_Large = zeros(Int, T)
N_Small = zeros(Int, T)
for k in 1:K
    if is_large_vessel[k]
        N_Large .+= N_Vessels[k]
    else
        N_Small .+= N_Vessels[k]
    end
end
N_Total = N_Large .+ N_Small
@assert length(N_Total) == T

@printf("Total arrivals in 24h : %d vessels\n", sum(N_Total))
@printf("  - Large Vessels     : %d\n", sum(N_Large))
@printf("  - Small Vessels     : %d\n", sum(N_Small))
@printf("Peak slot demand      : %d vessels/30-min\n\n", maximum(N_Total))

# ----------------------------------------------------------------
# 2. WEATHER & TIDE SCENARIOS (Probabilities p[q])
#
# Singapore Maritime Weather Context:
#   Clear / Good Weather: Optimal berthing and tide.
#   Low Tide / Swell: Draft constraints reduce availability of Large Berths.
#   Sumatra Squalls (Wind > 20 knots): Halts pilotage & berthing.
# ----------------------------------------------------------------
scenario_type = vcat(
    fill("Clear Weather",       8),
    fill("Low Tide / Swell",    7),
    fill("Sumatra Squall",      5)
)

raw_w = vcat([0.10, 0.08, 0.08, 0.06, 0.06, 0.05, 0.04, 0.03], #Clear(50%)
[0.07, 0.06, 0.06, 0.05, 0.04, 0.04, 0.03],   #Low Tide(35%)
[0.05, 0.04, 0.03, 0.02, 0.01]                # Squall (15%)
)
p = raw_w ./ sum(raw_w)

println("Scenario probabilities (Weather & Tide Impacts):")
@printf("  %-22s  %8s  %8s\n", "Type", "Scenarios", "P(total)")
for st in unique(scenario_type)
    qs   = findall(x -> x == st, scenario_type)
    ptot = sum(p[q] for q in qs)
    @printf("  %-22s  %8d  %8.3f\n", st, length(qs), ptot)
end

# ----------------------------------------------------------------
# 3. BERTHING "PARKING LOT" CAPACITY  M[q,i]
#
# We separate capacity into M_Large (Only fits Large, but Small can use if empty) 
# and M_Small (Only fits Small).
# 
# Base Available Lots (Dynamic per slot due to tug/pilot throughput):
#   - Clear Weather:  Large Lots = 2, Small Lots = 6
#   - Low Tide:       Large Lots = 2 (Draft restrictions), Small = 6
#   - Squall (>20kt): Large Lots = 0, Small = 1 (Pilotage suspended)
# ----------------------------------------------------------------
base_large_cap = Dict("Clear Weather" => 2.0, "Low Tide / Swell" => 1.0, "Sumatra Squall" => 0.0)
base_small_cap = Dict("Clear Weather" => 6.0, "Low Tide / Swell" => 6.0, "Sumatra Squall" => 0.0)

M_Large = zeros(Int, Q, T)
M_Small = zeros(Int, Q, T)
for q in 1:Q
   stype = scenario_type[q]

    # Generate dynamic time windows for weather events
    if stype == "Sumatra Squall"
        # Squalls typically hit pre-dawn or late afternoon in Singapore
        event_pool = rand() < 0.5 ? (8:14) : (30:38)
        event_start = rand(event_pool)
        event_len = rand(2:4) # Lasts 1 to 2 hours
    elseif stype == "Low Tide / Swell"
        # Tides affect cycles roughly 12 hours apart
        event_start = rand(1:10)
        event_len = rand(4:8)
    else
        event_start, event_len = 0, 0
    end

    for i in 1:T
        # Is the slot under an active weather/tide event?
        in_event = (event_start <= i < event_start + event_len)
        
        # Apply capacity: Default to clear weather unless an event is active
        cap_L = in_event ? base_large_cap[stype] : base_large_cap["Clear Weather"]
        cap_S = in_event ? base_small_cap[stype] : base_small_cap["Clear Weather"]
        
        
        # Add slight operational noise
        utilization = 0.6  # 60% realistic throughput
        M_Large[q,i] = max(0, round(Int, utilization * cap_L + (randn() * 0)))
        M_Small[q,i] = max(0, round(Int, utilization * cap_S + (randn() * 0)))
    end
end
# ================================================================
# 4. VESSEL KINEMATICS & KINETIC LIMITS (AIS DATA SIMULATION)
# To prove JIT works, we must calculate the maximum time a ship 
# can delay its arrival by reducing its speed before it stalls.
# ================================================================

# Simulated AIS tracking distance from Port (when JIT Command is issued)
# Deep sea vessels are tracked from 400-500nm away, local tugs from 50nm.
planning_distance_nm = [
    600.0,  # Bulk
    600.0,  # Tanker
    800.0,  # Container
    300.0,  
    80.0,   
    500.0,  
    150.0   
]

# Speeds in knots (1 knot = 1 nm per hour)
vessel_speed_design =[14.0, 15.0, 22.0, 13.0, 11.0, 20.0, 12.0]
vessel_speed_min    =[ 9.0,  9.0, 12.0,  8.0,  5.0, 12.0,  6.0]

max_jit_hrs = zeros(Float64, K)
println()
println("--- AIS Kinematics & Slow Steaming Limits ---")
@printf("%-15s | %-12s | %-10s | %-9s | %-15s\n", "Vessel Class", "Distance(nm)", "Cruise(kt)", "Min(kt)", "Max JIT Delay(hr)")
println("-"^70)
for k in 1:K
    # Time it takes if cruising normally
    time_normal = planning_distance_nm[k] / vessel_speed_design[k]
    # Time it takes if running at minimum safe engine RPM
    time_slowest = planning_distance_nm[k] / vessel_speed_min[k]
    
    # The maximum amount of delay they can "absorb" out in the ocean
    max_jit_hrs[k] = time_slowest - time_normal
    
    @printf("%-15s | %-12.1f | %-10.1f | %-9.1f | %5.1f hrs\n", 
            class_names[k], planning_distance_nm[k], vessel_speed_design[k], vessel_speed_min[k], max_jit_hrs[k])
end
println()


# ================================================================
# 5. COSTS (JIT vs Anchorage) - USING REAL-WORLD MPA DATA
#prove that "Just-In-Time" (JIT) arrivals and slow steaming save money and fuel compared to racing to port and sitting at anchor.
# ================================================================

# A. MPA Singapore Port Dues (Category 1: Cargo Operations)
# Rates are $ per 100 GT for the cumulative period of stay.
mpa_cat1_rates_per_100GT = Dict(1 => 8.00, 2 => 8.50, 3 => 9.00)

# B. Vessel Specifications by Class (Expanded to K = 7) TO BE EDITED]
# Average Gross Tonnage estimates for the classes arriving at SG
vessel_GT =[40_000, 80_000, 100_000, 20_000, 2_000, 120_000, 10_000] 

# C. Auxiliary engine fuel consumption at anchor (tons per day)
#GT = ship's total internal volume.
#TEU = # of cargo capacity (# of 20-foot containers)
#DWT = A measure of the weight a ship can safely carry
#1 TEU = 10-15 GT, 1.9 DWT = 1 GT
aux_fuel_tons_per_day =[4.0, 6.0, 8.0, 2.5, 1.0, 15.0, 2.0] #[TO BE EDITED]
#Refernce: https://ww2.arb.ca.gov/sites/default/files/barcu/regact/2008/fuelogv08/appdfuel.pdf
fuel_price_per_ton = 650.0 # USD per ton (VLSFO standard)
#Refernce: https://www.sea-intelligence.com/press-room/116-new-bunker-fuel-price-records

# D. Daily Charter Rate/Opportunity Cost (USD per day)[For simple model set to 0]
#Refernce: https://hz-containers.com/en/news/daily-rates-in-maritime-shipping/
#Refernce: https://www.rivieramm.com/news-content-hub/news-content-hub/global-maritime-trade-slows-as-tankers-and-bulk-carriers-face-challenges-85372
include_opportunity_cost = false 
charter_rate_per_day = include_opportunity_cost ? [20_000.0, 40_000.0, 50_000.0, 15_000.0, 5_000.0, 100_000.0, 10_000.0] : zeros(Float64, K)

# E. Calculate base Hourly Anchorage Cost (base_anch_hr)
base_anch_hr = zeros(Float64, K)

println("--- Calculating Realistic Anchorage Costs ---")
for k in 1:K
    # 1. MPA Port Dues (Using Day 1 rate for hourly breakdown)
    daily_mpa_dues = mpa_cat1_rates_per_100GT[1] * (vessel_GT[k] / 100.0)
    hourly_mpa_dues = daily_mpa_dues / 24.0
    # 2. Auxiliary Fuel
    daily_fuel_cost = aux_fuel_tons_per_day[k] * fuel_price_per_ton
    hourly_fuel_cost = daily_fuel_cost / 24.0
    # 3. Opportunity Cost (Lost charter earnings)
    hourly_charter_cost = charter_rate_per_day[k] / 24.0
    # Totals
    base_anch_hr[k] = hourly_mpa_dues + hourly_fuel_cost + hourly_charter_cost
    
    @printf("%-15s: MPA = \$%6.2f/hr | Fuel = \$%6.2f/hr | Charter = \$%7.2f/hr -> TOTAL = \$%7.2f/hr\n", 
            class_names[k], hourly_mpa_dues, hourly_fuel_cost, hourly_charter_cost, base_anch_hr[k])
end

# F. JIT (Slow Steaming) Cost Calculation
# JIT saves main engine fuel but incurs a commercial late penalty. 
# Base penalty is set slightly lower than Anchorage to incentivize slow steaming, 
# until 12 hours where minimum safe RPMs are hit and penalty explodes.
base_jit_hr = [
    400.0,   # Bulk
    900.0,   # Tanker
    1100.0,  # Container
    300.0,   # Cargo
    100.0,   # Tug
    2000.0,  # Passenger
    200.0    # Other
]  #[TBC]
#Refernce: about 5-10% cheaper then Physcial anchorage when delayed moderately, but much cheaper than anchorage 
    #if delayed excessively (since fuel burn is the main cost driver for JIT, while port dues and charter costs dominate Anchorage).

# Initialize matrices
c_Anch = zeros(Float64, K, T, T)
c_JIT  = zeros(Float64, K, T, T)
for k in 1:K, i in 1:T, j in i:T
    delay_hrs = (j - i) / 2.0

    # 1. Physical Anchorage Cost (Non-Linear to prevent port congestion)
    #Assume within the first 2 hours, costs are linear and predictable.
    #Assume within 6 hours, costs increase more steeply due to congestion, operational inefficiencies, and potential missed connections.
    #Assume After 6 hours, costs explode due to severe congestion, missed connections, and operational disruptions.
    if delay_hrs <= 2.0
        c_Anch[k, i, j] = base_anch_hr[k] * delay_hrs
    elseif delay_hrs <= 6.0
        # Multiplier kicks in to punish physical port queues
        c_Anch[k, i, j] = (base_anch_hr[k] * 2.0) + (base_anch_hr[k] * 1.5 * (delay_hrs - 2.0))
    else
        # Explosion in cost (missed connections, severe congestion)
        c_Anch[k, i, j] = (base_anch_hr[k] * 2.0) + (base_anch_hr[k] * 1.5 * 4.0) + (base_anch_hr[k] * 3.0 * (delay_hrs - 6.0))
    end
    
    # 2. Virtual JIT Cost(Linear/Cheaper, but physically limited by max_jit_hrs limit)
    #Assume 10,000 USD/hr penalty if JIT delay exceeds max_jit_hrs[hours] due to safety and operational limits of slow steaming.
    #Assume max_jit_hrs[hours] is the maximum feasible JIT delay before safety and operational issues arise (e.g. minimum safe RPMs, crew fatigue, schedule disruptions).
    if delay_hrs <= max_jit_hrs[k]
        c_JIT[k, i, j] = base_jit_hr[k] * delay_hrs
    else
        # Ship has slowed down to minimum RPM. Any further delay forces them to drop anchor.
        # We apply the max JIT cost + massive penalty for drifting / entering anchor state late
        c_JIT[k, i, j] = (base_jit_hr[k] * max_jit_hrs[k]) + (10_000.0 * (delay_hrs - max_jit_hrs[k]))
    end
end
# ----------------------------------------------------------------
# 4. SHOWCASE OPTIMIZER INCENTIVES (Anchorage vs JIT)
# ----------------------------------------------------------------
println("\nCost comparison based on delay duration (Example: Container Ship):")
k_ex = 3 # Container ship index
println(" Delay  | JIT Cost (Virtual) | Anchorage Cost (Physical) | Optimizer Choice")
println("----------------------------------------------------------------------------")
for d in [1.0, 3.0, 6.0, 14.0]
    i = 1; j = i + Int(d * 2)
    anch = c_Anch[k_ex, i, j]
    jit  = c_JIT[k_ex, i, j]
    choice = jit < anch ? "SLOW STEAM (Save Fuel & Queue)" : "PHYSICAL ANCHOR (Engine limits hit)"
    @printf(" %4.1f h | \$%15.2f | \$%23.2f | %s\n", d, jit, anch, choice)
end

println("\nData generation complete. Matrices `c_Anch` and `c_JIT` are ready for optimization.")


# ----------------------------------------------------------------
# 5. SANITY CHECKS & MODEL OUTPUT
# ----------------------------------------------------------------
println("\n" * "="^72)
println("DATA & CAPACITY SUMMARY")
println("="^72)
@printf("Problem size     : T=%d slots × Q=%d scenarios\n", T, Q)
@printf("Ship Classes     : Large (VLCC/Box), Small (Cargo/Tugs)\n")
println()
@printf("Peak 'Large' Demand   : %d vessels/slot\n", maximum(N_Large))
@printf("Peak 'Small' Demand   : %d vessels/slot\n", maximum(N_Small))
@printf("Clear Wx 'Large' Lots : %.0f lots available per slot\n", base_large_cap["Clear Weather"])
@printf("Clear Wx 'Small' Lots : %.0f lots available per slot\n", base_small_cap["Clear Weather"])
println()
@printf("Base Anchorage Cost (1st hr) - Tanker (Large) : \$%.0f\n", c_Anch[2, 1, 3])
@printf("Base Anchorage Cost (1st hr) - Tug (Small)    : \$%.0f\n", c_Anch[5, 1, 3])
println()

println("Average Capacity vs Demand feasibility check:")
@printf("  %-20s  %10s  %10s  %10s\n", "Scenario type", "AvgLargeCap", "AvgSmallCap", "DemandMet?")
for st in unique(scenario_type)
    qs   = findall(x -> x == st, scenario_type)
    q    = qs[1]
    avg_L = mean(M_Large[q,:])
    avg_S = mean(M_Small[q,:])
    
    tot_L_cap = sum(M_Large[q,:])
    tot_S_cap = sum(M_Small[q,:])
    
    # In optimization, small ships can use large slots, but large CANNOT use small slots
    L_ok = tot_L_cap >= sum(N_Large)
    S_ok = (tot_L_cap + tot_S_cap - sum(N_Large)) >= sum(N_Small)
    
    tag = (L_ok && S_ok) ? "OK" : "BOTTLENECK"
    @printf("  %-20s  %10.1f  %10.1f  %10s\n", st, avg_L, avg_S, tag)
end

println("\nTraffic Demand Distribution by Time of Day (Combined):")
for i in 1:4:T
    bar_L = repeat("█", N_Large[i])
    bar_S = repeat("▒", N_Small[i])
    @printf("  %s  %2d Total (L:%d, S:%d) | %s%s\n", 
            period_label[i], N_Total[i], N_Large[i], N_Small[i], bar_L, bar_S)
end
println("  (█ = Large Ship, ▒ = Small Ship)")

PORT OF SINGAPORE (MPA) — MARITIME TRAFFIC & BERTHING SCHEDULING
Total arrivals in 24h : 94 vessels
  - Large Vessels     : 71
  - Small Vessels     : 23
Peak slot demand      : 7 vessels/30-min

Scenario probabilities (Weather & Tide Impacts):
  Type                    Scenarios  P(total)
  Clear Weather                  8     0.500
  Low Tide / Swell               7     0.350
  Sumatra Squall                 5     0.150

--- AIS Kinematics & Slow Steaming Limits ---
Vessel Class    | Distance(nm) | Cruise(kt) | Min(kt)   | Max JIT Delay(hr)
----------------------------------------------------------------------
Bulk Carrier    | 600.0        | 14.0       | 9.0       |  23.8 hrs
Tanker          | 600.0        | 15.0       | 9.0       |  26.7 hrs
Container       | 800.0        | 22.0       | 12.0      |  30.3 hrs
Cargo           | 300.0        | 13.0       | 8.0       |  14.4 hrs
Tug/Supply      | 80.0         | 11.0       | 5.0       |   8.7 hrs
Passenger       | 500.0        | 20.0   

In [21]:
large_classes = findall(is_large_vessel)
println("\nLarge Vessel Classes in this dataset:", large_classes)
small_classes = findall(.!is_large_vessel)
println("Small Vessel Classes in this dataset:", small_classes)


Large Vessel Classes in this dataset:[1, 2, 3, 6]
Small Vessel Classes in this dataset:[4, 5, 7]


In [22]:
#Optimization model
using JuMP, HiGHS
model = Model(HiGHS.Optimizer)

# ================================================================
# DECISION VARIABLES
# X_Anch[k, i, j]: # of class-k vessels arriving at slot i, 
#                   assigned to PHYSICAL ANCHOR, berthing at slot j
# X_JIT[k, i, j]:  # of class-k vessels arriving at slot i, 
#                   assigned to SLOW STEAM (JIT), berthing at slot j
# W_Large[q, i]: backlog of LARGE vessels waiting at end of slot i, scenario q
# W_Small[q, i]: backlog of SMALL vessels waiting at end of slot i, scenario q
# ================================================================
@variable(model, X_Anch[k=1:K, i=1:T, j=i:T] >= 0, Int)
@variable(model, X_JIT[k=1:K,  i=1:T, j=i:T] >= 0, Int)
@variable(model, W_Large[q=1:Q, i=0:T] >= 0)
@variable(model, W_Small[q=1:Q, i=0:T] >= 0)

# ================================================================
# EXPRESSIONS
# Total vessels of class k assigned to berth at slot j (from any earlier arrival slot i)
# This is the "throughput" arriving at the berth gate at slot j
# ================================================================
@expression(model, Berth_Large[j=1:T],
    sum(
        (X_Anch[k, i, j] + X_JIT[k, i, j])
        for k in large_classes
        for i in 1:j
    )
)

@expression(model, Berth_Small[j=1:T],
    sum(
        (X_Anch[k, i, j] + X_JIT[k, i, j])
        for k in small_classes
        for i in 1:j
    )
)

# ================================================================
# CONSTRAINTS
# ================================================================

# 1. DEMAND SATISFACTION: Every vessel that arrives at slot i 
#    must be assigned to exactly one berthing slot j >= i
#    (either via Anchorage or JIT)
@constraint(model, Demand[k=1:K, i=1:T],
    sum(X_Anch[k,i,j] + X_JIT[k,i,j] for j in i:T) == N_Vessels[k][i]
)

# 2. BACKLOG INITIAL CONDITIONS
@constraint(model, [q=1:Q], W_Large[q, 0] == 0)
@constraint(model, [q=1:Q], W_Small[q, 0] == 0)

# 3. BACKLOG EVOLUTION: Unserved vessels spill into next slot's queue
#    Large berths: only large vessels consume large berth capacity
#    Small vessels can ALSO use leftover large berth capacity
@constraint(model, BacklogLarge[q=1:Q, i=1:T],
    W_Large[q, i] >= W_Large[q, i-1] + Berth_Large[i] - M_Large[q, i]
)

# Small vessels use M_Small capacity only
# (spillover from large berths is handled implicitly via backlog penalty)
@constraint(model, BacklogSmall[q=1:Q, i=1:T],
    W_Small[q, i] >= W_Small[q, i-1] + Berth_Small[i] - M_Small[q, i]
)

# 4. JIT KINEMATIC LIMIT: A vessel cannot slow-steam beyond max_jit_hrs[k]
#    If j - i > max_jit_hrs[k] * 2 slots, JIT is infeasible → force to Anchorage
@constraint(model, JITLimit[k=1:K, i=1:T, j=i:T;
    (j - i) / 2.0 > max_jit_hrs[k]],
    X_JIT[k, i, j] == 0
)

# ================================================================
# OBJECTIVE
# OBJECTIVE: Minimise total expected cost
#   = Direct vessel delay costs (Anchorage + JIT)
#   + Penalty C on expected backlog across all scenarios
# ================================================================
C = 50_000.0 # Backlog penalty per vessel-slot (USD) — tune as needed

@objective(model, Min,
    # Term 1: Vessel delay costs (deterministic, summed over all classes)
    sum(
        c_Anch[k, i, jj] * X_Anch[k, i, jj] +
        c_JIT[k, i, jj]  * X_JIT[k, i, jj]
        for k in 1:K, i in 1:T, jj in i:T
    )
    #Term 2: Expected backlog penalty across weather scenarios
    + C * sum(
        p[q] * (W_Large[q, i] + W_Small[q, i])
        for q in 1:Q, i in 1:T
    )
)

optimize!(model)


Running HiGHS 1.13.1 (git hash: 1d267d97c): Copyright (c) 2026 under Apache 2.0 license terms
Using BLAS: blastrampoline-5 
MIP has 3309 rows; 18424 cols; 350637 nonzeros; 16464 integer variables (0 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+01, 1e+05]
  Bound   [0e+00, 0e+00]
  RHS     [1e+00, 4e+00]
Presolving model
1956 rows, 5141 cols, 72642 nonzeros  0s
1551 rows, 3070 cols, 35028 nonzeros  0s
Presolve reductions: rows 1551(-1758); columns 3070(-15354); nonzeros 35028(-315609) 

Solving MIP model with:
   1551 rows
   3070 cols (1292 binary, 299 integer, 0 implied int., 1479 continuous, 0 domain fixed)
   35028 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u =

In [23]:

# ================================================================
# RESULTS
# ================================================================
println("\n", "="^72)
println("OPTIMIZATION RESULTS")
println("="^72)
@printf("Solver status  : %s\n", termination_status(model))
@printf("Objective value: \$%.2f\n", objective_value(model))

total_anch = sum(value(X_Anch[kk,ii,jj]) for kk in 1:K, ii in 1:T, jj in 1:T if jj >= ii)
total_jit  = sum(value(X_JIT[kk,ii,jj])  for kk in 1:K, ii in 1:T, jj in 1:T if jj >= ii)
@printf("Vessels via Anchorage : %.0f\n", total_anch)
@printf("Vessels via JIT       : %.0f\n", total_jit)
@printf("JIT adoption rate     : %.1f%%\n", 100 * total_jit / (total_anch + total_jit))

println("Total Large Demand: ", sum(N_Large))
println("Total Large Capacity (Clear): ", sum(M_Large[1,:]))


OPTIMIZATION RESULTS
Solver status  : OPTIMAL
Objective value: $1796315.62
Vessels via Anchorage : 78
Vessels via JIT       : 16
JIT adoption rate     : 17.0%
Total Large Demand: 71
Total Large Capacity (Clear): 48
